In [6]:
import pandas as pd

# Load kedua CSV
wikidata = pd.read_csv("brandproductindustry_wikidata.csv")
dbpedia = pd.read_csv("brandproductindustry_dbpedia.csv")

# Ambil label doang
wikidata = wikidata[['brandLabel', 'productLabel', 'industryLabel', 'countryLabel']]
dbpedia = dbpedia[['brandLabel', 'productLabel', 'industryLabel', 'countryLabel']]

# Normalisasi teks
for df in [wikidata, dbpedia]:
    df['brandLabel'] = df['brandLabel'].str.strip().str.lower()
    df['productLabel'] = df['productLabel'].str.strip().str.lower()
    df['industryLabel'] = df['industryLabel'].str.strip().str.lower()
    df['countryLabel'] = df['countryLabel'].str.strip().str.lower()

# Tambah kolom sumber
wikidata['source'] = 'wikidata'
dbpedia['source'] = 'dbpedia'

# Gabung outer join
merged = pd.merge(
    wikidata,
    dbpedia,
    on=['brandLabel', 'productLabel'],
    how='outer',
    suffixes=('_wiki', '_dbp')
)

# Fallback: pakai wikidata, kalau kosong pakai dbpedia
merged['industryLabel'] = merged['industryLabel_wiki'].combine_first(merged['industryLabel_dbp'])
merged['countryLabel'] = merged['countryLabel_wiki'].combine_first(merged['countryLabel_dbp'])

# Buang kolom duplikat dan source
merged = merged.drop(columns=[
    'industryLabel_wiki', 'industryLabel_dbp',
    'countryLabel_wiki', 'countryLabel_dbp',
    'source_wiki', 'source_dbp' # Corrected column names to drop
], errors='ignore')

# Buang baris kosong dan duplikat
merged = merged.dropna(subset=['brandLabel', 'productLabel'])
merged = merged.drop_duplicates(subset=['brandLabel', 'productLabel'])

# Kapitalisasi
for col in ['brandLabel', 'productLabel', 'industryLabel', 'countryLabel']:
    merged[col] = merged[col].str.title()

print(f"Total baris: {len(merged)}")
print(merged.head(10))

# Simpan
merged.to_csv("dataset_final.csv", index=False)

Total baris: 23108
                 brandLabel             productLabel           industryLabel  \
0               "Pro:Atria"                 Sftpplus       Software Industry   
1                .224-32 Fa                   Pistol                 Firearm   
2                .224-32 Fa                 Revolver                 Firearm   
3   .Nelson Publishers Ltd.                     Book              Publishing   
4                    1&1 Ag  Mobile Network Operator   Mobile Phone Industry   
6                 1+1 Media       Television Program          Media Industry   
7             1-800-Flowers                   Flower                  Retail   
8               1-Up Studio         Magical Starsign     Video Game Industry   
9               1-Up Studio                 Mother 3     Video Game Industry   
10         10 Minute School               Mobile App  Educational Technology   

      countryLabel  
0   United Kingdom  
1    United States  
2    United States  
3            Nig